# Проверка FastAPI-сервиса загрузкой презентации

Этот notebook отправляет `PPTX` и при необходимости `PDF` в ручку `POST /presentations`.
Никакая командная строка не нужна: достаточно заполнить переменные и выполнить ячейки сверху вниз.

In [ ]:
from __future__ import annotations

import json
import mimetypes
import uuid
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

In [ ]:
BASE_URL = "http://127.0.0.1:8000"
PPTX_PATH = Path(r"C:\\path\\to\\presentation.pptx")
PDF_PATH = None
ADDITIONAL_INFO = ""
REPORT_NAME = None
PRESENTATION_ID = None
TIMEOUT_SECONDS = 300

Если у тебя есть PDF-версия той же презентации, задай `PDF_PATH = Path(r"...")`.
Если PDF не нужен, оставь `PDF_PATH = None`.

In [ ]:
def build_multipart_body(*, fields: dict[str, str], files: dict[str, Path], boundary: str) -> bytes:
    """Собирает тело multipart/form-data из текстовых полей и файлов."""
    line_break = b"\r\n"
    body_parts: list[bytes] = []

    for field_name, field_value in fields.items():
        body_parts.append(f"--{boundary}".encode("utf-8"))
        body_parts.append(f'Content-Disposition: form-data; name="{field_name}"'.encode("utf-8"))
        body_parts.append(b"")
        body_parts.append(field_value.encode("utf-8"))

    for field_name, file_path in files.items():
        content_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
        body_parts.append(f"--{boundary}".encode("utf-8"))
        body_parts.append(
            (
                f'Content-Disposition: form-data; name="{field_name}"; '
                f'filename="{file_path.name}"'
            ).encode("utf-8")
        )
        body_parts.append(f"Content-Type: {content_type}".encode("utf-8"))
        body_parts.append(b"")
        body_parts.append(file_path.read_bytes())

    body_parts.append(f"--{boundary}--".encode("utf-8"))
    body_parts.append(b"")
    return line_break.join(body_parts)


def validate_paths(pptx_path: Path, pdf_path: Path | None) -> None:
    """Проверяет, что файлы существуют и имеют ожидаемые расширения."""
    if not pptx_path.exists() or not pptx_path.is_file():
        raise FileNotFoundError(f"Не найден PPTX-файл: {pptx_path}")
    if pptx_path.suffix.lower() != ".pptx":
        raise ValueError(f"Файл `{pptx_path}` должен иметь расширение `.pptx`.")

    if pdf_path is None:
        return
    if not pdf_path.exists() or not pdf_path.is_file():
        raise FileNotFoundError(f"Не найден PDF-файл: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"Файл `{pdf_path}` должен иметь расширение `.pdf`.")


def upload_presentation(
    *,
    base_url: str,
    pptx_path: Path,
    pdf_path: Path | None,
    additional_info: str,
    report_name: str | None,
    presentation_id: str | None,
    timeout_seconds: int,
) -> tuple[int, str]:
    """Отправляет презентацию в FastAPI-сервис и возвращает HTTP-статус и тело ответа."""
    boundary = f"----presentation-upload-{uuid.uuid4().hex}"
    fields = {"additional_info": additional_info}
    if report_name:
        fields["report_name"] = report_name
    if presentation_id:
        fields["presentation_id"] = presentation_id

    files = {"pptx_file": pptx_path}
    if pdf_path is not None:
        files["pdf_file"] = pdf_path

    body = build_multipart_body(fields=fields, files=files, boundary=boundary)
    request = Request(
        url=f"{base_url.rstrip('/')}/presentations",
        data=body,
        headers={
            "Content-Type": f"multipart/form-data; boundary={boundary}",
            "Content-Length": str(len(body)),
        },
        method="POST",
    )

    try:
        with urlopen(request, timeout=timeout_seconds) as response:
            return response.getcode(), response.read().decode("utf-8")
    except HTTPError as error:
        return error.code, error.read().decode("utf-8", errors="replace")
    except URLError as error:
        raise RuntimeError(
            f"Не удалось подключиться к сервису по адресу `{base_url}`: {error.reason}"
        ) from error

In [ ]:
validate_paths(PPTX_PATH, PDF_PATH)
print("PPTX:", PPTX_PATH)
print("PDF:", PDF_PATH)
print("BASE_URL:", BASE_URL)

In [ ]:
status_code, response_text = upload_presentation(
    base_url=BASE_URL,
    pptx_path=PPTX_PATH,
    pdf_path=PDF_PATH,
    additional_info=ADDITIONAL_INFO,
    report_name=REPORT_NAME,
    presentation_id=PRESENTATION_ID,
    timeout_seconds=TIMEOUT_SECONDS,
)

print(f"HTTP {status_code}")
try:
    print(json.dumps(json.loads(response_text), ensure_ascii=False, indent=2))
except json.JSONDecodeError:
    print(response_text)